# R2-CAP · Capstone — Profundización (opcional) 🔬

**Formación Pública — Rama R2 · Científico de Datos · Notebook de profundización**

Opcional. Tres refuerzos para que tu capstone resista preguntas difíciles: **validación cruzada
temporal** (¿cuánto varía el desempeño?), **desempeño por estación** (¿a quién le falla?) y el
**trade-off recall/precision** (¿cuántas falsas alarmas aceptamos para no perder episodios?).
Mismo ejemplo de PM2.5; aplícalo a tus datos.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import f1_score, recall_score, precision_score

CSV = "aire_diario.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{CSV}", CSV)
df = pd.read_csv(CSV, parse_dates=["fecha"])

def construir_tabla(df):
    partes = []
    for estacion, d in df.groupby("estacion"):
        d = d.set_index("fecha").sort_index().asfreq("D")
        d["estacion"] = estacion
        d["pm25_ayer"] = d["pm25"].shift(1)
        d["media_7d"] = d["pm25"].rolling(7, min_periods=5).mean()
        d["mes"] = d.index.month
        d["pm25_manana"] = d["pm25"].shift(-1)
        partes.append(d.reset_index())
    t = pd.concat(partes, ignore_index=True)
    t = t.dropna(subset=["pm25", "pm25_ayer", "media_7d", "pm25_manana"]).reset_index(drop=True)
    t["episodio_manana"] = (t["pm25_manana"] > 50).astype(int)
    return t

tabla = construir_tabla(df).sort_values("fecha").reset_index(drop=True)   # orden temporal
FEATURES = ["estacion", "pm25", "pm25_ayer", "media_7d", "mes"]
en_prueba = tabla["fecha"] >= "2025-01-01"
Xtr, ytr = tabla.loc[~en_prueba, FEATURES], tabla.loc[~en_prueba, "episodio_manana"]
Xte, yte = tabla.loc[en_prueba, FEATURES], tabla.loc[en_prueba, "episodio_manana"]

def nuevo_pipeline(**kw):
    pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["estacion", "mes"]),
                             ("num", StandardScaler(), ["pm25", "pm25_ayer", "media_7d"])])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, **kw))])

print("entrenamiento:", Xtr.shape, "| prueba:", Xte.shape)

## 1. Validación cruzada temporal

Un solo año de prueba puede ser un año raro. `TimeSeriesSplit` hace varias particiones **respetando el orden**: cada pliegue entrena con el pasado y evalúa en el bloque siguiente. La dispersión entre pliegues dice cuánto confiar en una sola cifra.

### ✍️ Ejercicio 1 — F1 en 5 pliegues temporales

In [ ]:
scores = cross_val_score(nuevo_pipeline(), Xtr, ytr, cv=TimeSeriesSplit(n_splits=5), scoring="f1")
print("F1 por pliegue:", scores.round(3), f"| media {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert len(scores) == 5
    _s = cross_val_score(nuevo_pipeline(), Xtr, ytr, cv=TimeSeriesSplit(n_splits=5), scoring="f1")
    assert np.allclose(scores, _s)
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'cross_val_score(nuevo_pipeline(), Xtr, ytr, cv=TimeSeriesSplit(n_splits=5), scoring="f1").')
    print("   Detalle:", e)

## 2. ¿Para quién funciona peor? Recall por estación

Un buen promedio puede esconder estaciones donde el modelo casi no detecta episodios. En política pública eso es **equidad territorial**: ¿qué comunas quedan peor avisadas? Calcula el *recall* por estación en 2025, solo para estaciones con al menos 10 episodios.

### ✍️ Ejercicio 2 — Recall por estación

In [ ]:
modelo = nuevo_pipeline().fit(Xtr, ytr)
prueba = tabla.loc[en_prueba].assign(pred=modelo.predict(Xte))
con_episodios = prueba.groupby("estacion").filter(lambda d: d["episodio_manana"].sum() >= 10)
recall_estacion = con_episodios.groupby("estacion").apply(
    lambda d: recall_score(d["episodio_manana"], d["pred"]), include_groups=False).sort_values()
estacion_peor = recall_estacion.idxmin()
print(recall_estacion.round(2))
print("Peor avisada:", estacion_peor)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _r = {e: recall_score(d["episodio_manana"], d["pred"]) for e, d in con_episodios.groupby("estacion")}
    assert len(recall_estacion) == len(_r)
    assert estacion_peor == min(_r, key=_r.get)
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'con_episodios.groupby("estacion").apply(lambda d: recall_score(d["episodio_manana"], d["pred"]), include_groups=False); .idxmin().')
    print("   Detalle:", e)

## 3. El trade-off: pesar más los episodios

`class_weight="balanced"` le dice al modelo que equivocarse en un episodio (la clase rara) cuesta más. Sube el *recall*… y baja la *precision* (más falsas alarmas). **¿Cuál conviene?** No es una pregunta técnica: depende de cuánto cuesta un episodio no avisado frente a una alarma innecesaria. Mide ambos modelos.

### ✍️ Ejercicio 3 — Recall y precision con y sin balanceo

In [ ]:
bal = nuevo_pipeline(class_weight="balanced").fit(Xtr, ytr)
p0, p1 = modelo.predict(Xte), bal.predict(Xte)
comparacion = pd.DataFrame({
    "recall": [recall_score(yte, p0), recall_score(yte, p1)],
    "precision": [precision_score(yte, p0), precision_score(yte, p1)],
    "alertas": [int(p0.sum()), int(p1.sum())],
}, index=["sin balanceo", "balanced"]).round(3)
print(comparacion)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    _b = nuevo_pipeline(class_weight="balanced").fit(Xtr, ytr).predict(Xte)
    assert abs(float(comparacion.loc["balanced", "recall"]) - recall_score(yte, _b)) < 1e-3
    assert float(comparacion.loc["balanced", "recall"]) > float(comparacion.loc["sin balanceo", "recall"])
    assert int(comparacion.loc["balanced", "alertas"]) == int(_b.sum())
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'bal = nuevo_pipeline(class_weight="balanced").fit(Xtr, ytr); arma un DataFrame con recall_score, precision_score y .sum() de cada predicción.')
    print("   Detalle:", e)

---

*Fin de la profundización del capstone R2.*